# UNSW-NB15: чи не є ΔAP (F1−F0) артефактом early stopping?
1) Відтворюємо заморожений результат статті (F0 test_AP=0.973652, F1=0.983969) — санітарна перевірка пайплайну.
2) Порівнюємо F0 і F1 при ОДНАКОВІЙ кількості дерев (у статті F0 зупинився на 33, F1 — на 290).

In [1]:
import pandas as pd, numpy as np, lightgbm as lgb, json, os
from sklearn.metrics import average_precision_score, roc_auc_score

PATH = "/srv/ids_research/data/enriched/unsw_nb15_with_cve_context_dedup.parquet"
TS_COL = "Stime"
SEED = 42

F0_UNSW = ['state','dur','sbytes','dbytes','sttl','dttl','sloss','dloss','Sload','Dload','Spkts','Dpkts',
'swin','dwin','stcpb','dtcpb','smeansz','dmeansz','trans_depth','res_bdy_len','Sjit','Djit','Sintpkt',
'Dintpkt','tcprtt','synack','ackdat','is_sm_ips_ports','ct_state_ttl','ct_flw_http_mthd','is_ftp_login',
'ct_ftp_cmd','ct_srv_src','ct_srv_dst','ct_dst_ltm','ct_src_ltm','ct_src_dport_ltm','ct_dst_sport_ltm',
'ct_dst_src_ltm','proto','service']
F1_UNSW = F0_UNSW + ['cve_ctx_service','cve_ctx_pool_size','cve_ctx_epss_coverage',
                     'cve_ctx_n_generic_os_matches','cve_ctx_n_specific_product_matches']
CAT_COLS = {"state", "proto", "service", "cve_ctx_service"}

df = pd.read_parquet(PATH)
df.columns = [c.strip() for c in df.columns]
print("rows:", len(df), "| cols:", len(df.columns))
print("відсутні колонки:", [c for c in F1_UNSW + ['Label', TS_COL] if c not in df.columns] or "— жодної")

# ct_ftp_cmd: ' ' -> NaN, далі to_numeric (як у замороженому протоколі)
df["ct_ftp_cmd"] = pd.to_numeric(df["ct_ftp_cmd"].replace(" ", np.nan), errors="coerce")
y = pd.to_numeric(df["Label"], errors="coerce").astype(int)

rows: 2059418 | cols: 62
відсутні колонки: — жодної


In [2]:
# Звірка з реальними контрактами ознак на сервері (якщо файли є)
contract_dir = "/home/ivan_r_opirskyy/notebooks/2 article/release/results/step5d2_audit"
for name, hardcoded in [("feature_contract_UNSW_F0.json", F0_UNSW), ("feature_contract_UNSW_F1.json", F1_UNSW)]:
    path = os.path.join(contract_dir, name)
    if not os.path.exists(path):
        print(f"[!] {name} не знайдено — звірку пропущено"); continue
    real = json.load(open(path))
    if isinstance(real, dict):
        lists = [v for v in real.values() if isinstance(v, list)]
        real = lists[0] if len(lists) == 1 else None
    if real is None:
        print(f"[?] {name}: структуру не визначено"); continue
    print(f"[{'OK' if list(real) == list(hardcoded) else 'MISMATCH'}] {name}: сервер={len(real)}, хардкод={len(hardcoded)}")

[OK] feature_contract_UNSW_F0.json: сервер=41, хардкод=41
[OK] feature_contract_UNSW_F1.json: сервер=46, хардкод=46


In [3]:
# Спліт як у статті: quantile 0.70/0.85 на Stime, snap до найближчого наявного значення
t = pd.to_numeric(df[TS_COL])
b1 = t.quantile(0.70, interpolation="nearest"); b2 = t.quantile(0.85, interpolation="nearest")
split = pd.Series(np.where(t < b1, "train", np.where(t < b2, "val", "test")), index=df.index)
tr, va, te = (split == "train"), (split == "val"), (split == "test")
print("межі:", pd.to_datetime(b1, unit='s'), "|", pd.to_datetime(b2, unit='s'),
      "  (у статті: 2015-02-18 05:36:51 | 08:58:36)")
print("розміри:", int(tr.sum()), int(va.sum()), int(te.sum()), "  (у статті: 1,441,568 / 308,915 / 308,935)")
print("AP випадкового класифікатора на test:", round(float(y[te].mean()), 6))

def prep(cols):
    frame = df[cols].copy()
    cats = [c for c in CAT_COLS if c in cols]
    for c in cats:
        frame[c] = pd.Categorical(frame[c], categories=pd.Categorical(frame.loc[tr, c]).categories)
    return frame, cats

def ap_feval(preds, data):
    return "unweighted_ap", average_precision_score(data.get_label(), preds), True

SPW = (y[tr] == 0).sum() / y[tr].sum()
PARAMS = dict(objective="binary", metric="None", seed=SEED, num_leaves=15, learning_rate=0.05,
              min_data_in_leaf=50, verbosity=-1, scale_pos_weight=SPW)
print("scale_pos_weight:", round(float(SPW), 4), "  (у статті: 28.2135)")

межі: 2015-02-18 05:36:51 | 2015-02-18 08:58:36   (у статті: 2015-02-18 05:36:51 | 08:58:36)
розміри: 1441568 308915 308935   (у статті: 1,441,568 / 308,915 / 308,935)
AP випадкового класифікатора на test: 0.082069
scale_pos_weight: 28.2135   (у статті: 28.2135)


In [4]:
# 1) Відтворення результату статті (early stopping на val)
def train_es(cols, tag):
    frame, cats = prep(cols)
    dtr = lgb.Dataset(frame[tr], label=y[tr], categorical_feature=cats, free_raw_data=False)
    dva = lgb.Dataset(frame[va], label=y[va], categorical_feature=cats, reference=dtr, free_raw_data=False)
    m = lgb.train(PARAMS, dtr, num_boost_round=500, valid_sets=[dva], feval=ap_feval,
                  callbacks=[lgb.early_stopping(30, verbose=False)])
    p = m.predict(frame[te], num_iteration=m.best_iteration)
    ap = average_precision_score(y[te], p)
    print(f"[{tag}] best_iter={m.best_iteration} test_AP={ap:.6f} test_AUC={roc_auc_score(y[te], p):.6f}")
    return ap, m.best_iteration

ap0, it0 = train_es(F0_UNSW, "F0 early-stop")
ap1, it1 = train_es(F1_UNSW, "F1 early-stop")
print(f"ΔAP = {ap1 - ap0:+.6f}   (у статті: F0=0.973652 @33, F1=0.983969 @290, ΔAP=+0.010317)")

[F0 early-stop] best_iter=33 test_AP=0.973652 test_AUC=0.997671
[F1 early-stop] best_iter=290 test_AP=0.983969 test_AUC=0.998247
ΔAP = +0.010317   (у статті: F0=0.973652 @33, F1=0.983969 @290, ΔAP=+0.010317)


In [5]:
# 2) F0 vs F1 при ОДНАКОВІЙ кількості дерев (без early stopping)
frames = {k: prep(c) for k, c in [("F0", F0_UNSW), ("F1", F1_UNSW)]}
models = {}
for k, (frame, cats) in frames.items():
    models[k] = lgb.train(PARAMS, lgb.Dataset(frame[tr], label=y[tr], categorical_feature=cats),
                          num_boost_round=500)
rows = []
for r in [33, 50, 100, 200, 290, 400, 500]:
    a = {k: average_precision_score(y[te], models[k].predict(frames[k][0][te], num_iteration=r)) for k in models}
    rows.append({"rounds": r, "AP_F0": a["F0"], "AP_F1": a["F1"], "dAP": a["F1"] - a["F0"]})
    print(f"rounds={r:>3}  AP_F0={a['F0']:.6f}  AP_F1={a['F1']:.6f}  dAP={a['F1'] - a['F0']:+.6f}")
out = pd.DataFrame(rows)
out.to_csv("/srv/ids_research/data/unsw_fixed_rounds_check.csv", index=False)
print("\nЗбережено: unsw_fixed_rounds_check.csv")

rounds= 33  AP_F0=0.973652  AP_F1=0.973858  dAP=+0.000206
rounds= 50  AP_F0=0.971914  AP_F1=0.977164  dAP=+0.005250
rounds=100  AP_F0=0.977313  AP_F1=0.980958  dAP=+0.003645
rounds=200  AP_F0=0.981662  AP_F1=0.983877  dAP=+0.002215
rounds=290  AP_F0=0.981978  AP_F1=0.983969  dAP=+0.001991
rounds=400  AP_F0=0.982266  AP_F1=0.984015  dAP=+0.001748
rounds=500  AP_F0=0.982439  AP_F1=0.983854  dAP=+0.001415

Збережено: unsw_fixed_rounds_check.csv


In [6]:
R_LIST = [290, 500]
N_BOOT = 500
y_te = y[te].to_numpy()
preds = {(k, r): models[k].predict(frames[k][0][te], num_iteration=r) for k in models for r in R_LIST}
minute = (pd.to_numeric(df.loc[te, TS_COL]).to_numpy() // 60).astype(np.int64)
blocks = pd.Series(np.arange(len(minute))).groupby(minute).indices
block_list = list(blocks.values())
print("рядків у test:", len(y_te), "| хвилинних блоків:", len(block_list))

rng = np.random.default_rng(SEED)
ci_rows = []
for r in R_LIST:
    p0, p1 = preds[("F0", r)], preds[("F1", r)]
    point = average_precision_score(y_te, p1) - average_precision_score(y_te, p0)
    simple, block = [], []
    for _ in range(N_BOOT):
        i = rng.integers(0, len(y_te), len(y_te))
        simple.append(average_precision_score(y_te[i], p1[i]) - average_precision_score(y_te[i], p0[i]))
        b = rng.integers(0, len(block_list), len(block_list))
        j = np.concatenate([block_list[k] for k in b])
        block.append(average_precision_score(y_te[j], p1[j]) - average_precision_score(y_te[j], p0[j]))
    for name, arr in [("simple", simple), ("block(minute)", block)]:
        lo, hi = np.percentile(arr, [2.5, 97.5])
        share_pos = float(np.mean(np.array(arr) > 0))
        ci_rows.append({"rounds": r, "bootstrap": name, "dAP": point, "CI95_lo": lo, "CI95_hi": hi, "P(dAP>0)": share_pos})
        print(f"rounds={r} {name:<14} dAP={point:+.6f}  95% CI [{lo:+.6f}, {hi:+.6f}]  частка реплік з dAP>0: {share_pos:.3f}")
pd.DataFrame(ci_rows).to_csv("/srv/ids_research/data/unsw_dAP_bootstrap_ci.csv", index=False)

рядків у test: 308935 | хвилинних блоків: 200
rounds=290 simple         dAP=+0.001991  95% CI [+0.001653, +0.002336]  частка реплік з dAP>0: 1.000
rounds=290 block(minute)  dAP=+0.001991  95% CI [+0.001289, +0.002835]  частка реплік з dAP>0: 1.000
rounds=500 simple         dAP=+0.001415  95% CI [+0.001019, +0.001835]  частка реплік з dAP>0: 1.000
rounds=500 block(minute)  dAP=+0.001415  95% CI [+0.000766, +0.002109]  частка реплік з dAP>0: 1.000


In [7]:
cat_te = df.loc[te, "attack_cat"].astype(str).str.strip().to_numpy()
normal = (cat_te == "Normal")
EXPLOIT_GROUP = {"Exploits", "Shellcode", "Worms"}
CONTROL_GROUP = {"DoS", "Fuzzers", "Generic", "Reconnaissance", "Analysis"}
R_PRIMARY = 500
p0, p1 = preds[("F0", R_PRIMARY)], preds[("F1", R_PRIMARY)]

rows = []
for cls in sorted(set(cat_te) - {"Normal"}):
    pos = (cat_te == cls)
    if pos.sum() == 0: continue
    m = pos | normal
    yy = pos[m].astype(int)
    a0, a1 = average_precision_score(yy, p0[m]), average_precision_score(yy, p1[m])
    grp = "exploit" if cls in EXPLOIT_GROUP else ("control" if cls in CONTROL_GROUP else "other")
    rows.append({"attack_cat": cls, "group": grp, "n_test": int(pos.sum()), "AP_F0": a0, "AP_F1": a1,
                 "dAP": a1 - a0, "err_reduction_%": 100 * (a1 - a0) / (1 - a0) if a0 < 1 else np.nan})
per_cls = pd.DataFrame(rows).sort_values(["group", "attack_cat"])
print(per_cls.round(6).to_string(index=False))

print("\nСередні по групах (зважені на n_test):")
for g, sub in per_cls.groupby("group"):
    w = sub["n_test"] / sub["n_test"].sum()
    print(f"  {g:<8} dAP={float((sub['dAP'] * w).sum()):+.6f}  err_reduction={float((sub['err_reduction_%'] * w).sum()):+.2f}%")
per_cls.to_csv("/srv/ids_research/data/unsw_per_class_dAP.csv", index=False)
print("\nЗбережено: unsw_per_class_dAP.csv, unsw_dAP_bootstrap_ci.csv")

    attack_cat   group  n_test    AP_F0    AP_F1       dAP  err_reduction_%
      Analysis control     621 0.843715 0.851465  0.007749         4.958364
           DoS control    1476 0.950880 0.959836  0.008956        18.232687
       Fuzzers control    4972 0.681966 0.708381  0.026415         8.305580
       Generic control    6902 0.996734 0.997983  0.001250        38.261832
Reconnaissance control    3448 0.987311 0.998570  0.011259        88.731665
      Exploits exploit    6896 0.980883 0.982435  0.001552         8.118136
     Shellcode exploit     373 0.683678 0.650143 -0.033535       -10.601514
         Worms exploit      42 0.742468 0.763205  0.020737         8.052271
      Backdoor   other     624 0.967494 0.974883  0.007389        22.732412

Середні по групах (зважені на n_test):
  control  dAP=+0.011299  err_reduction=+36.82%
  exploit  dAP=-0.000128  err_reduction=+7.16%
  other    dAP=+0.007389  err_reduction=+22.73%

Збережено: unsw_per_class_dAP.csv, unsw_dAP_bootstrap_ci

In [8]:
# 4b) Блочний bootstrap CI для ΔAP класу Exploits (первинна гіпотеза)
pos = (cat_te == "Exploits"); m_idx = np.where(pos | normal)[0]
keep = np.zeros(len(cat_te), bool); keep[m_idx] = True
block_list_m = [b[keep[b]] for b in block_list]
block_list_m = [b for b in block_list_m if len(b)]
yy_full = pos.astype(int)
vals = []
for _ in range(N_BOOT):
    b = rng.integers(0, len(block_list_m), len(block_list_m))
    j = np.concatenate([block_list_m[k] for k in b])
    if yy_full[j].sum() == 0: continue
    vals.append(average_precision_score(yy_full[j], p1[j]) - average_precision_score(yy_full[j], p0[j]))
lo, hi = np.percentile(vals, [2.5, 97.5])
print(f"Exploits @ {R_PRIMARY} дерев: dAP 95% block-CI [{lo:+.6f}, {hi:+.6f}], частка реплік з dAP>0: {np.mean(np.array(vals) > 0):.3f}")

Exploits @ 500 дерев: dAP 95% block-CI [+0.000317, +0.002874], частка реплік з dAP>0: 0.988
